In [8]:
import pandas as pd
import numpy as np
import zipfile
import os
from pathlib import Path
import torch


In [9]:
with   zipfile.ZipFile(r'C:\Users\PC\Downloads\sms+spam+collection.zip','r') as dataframe:
     print(dataframe.namelist())

     with dataframe.open('SMSSpamCollection') as f :
        df=pd.read_csv(f,
                      sep='\t',
                      header=None,
                      names=['labels','text'])


['SMSSpamCollection', 'readme']


In [10]:
print(df.head(9))

  labels                                               text
0    ham  Go until jurong point, crazy.. Available only ...
1    ham                      Ok lar... Joking wif u oni...
2   spam  Free entry in 2 a wkly comp to win FA Cup fina...
3    ham  U dun say so early hor... U c already then say...
4    ham  Nah I don't think he goes to usf, he lives aro...
5   spam  FreeMsg Hey there darling it's been 3 week's n...
6    ham  Even my brother is not like to speak with me. ...
7    ham  As per your request 'Melle Melle (Oru Minnamin...
8   spam  WINNER!! As a valued network customer you have...


In [11]:
df.value_counts('labels')

labels
ham     4825
spam     747
Name: count, dtype: int64

In [12]:
# blaced data d=swet for finetuing
def crate_blanced_dataset(df):

    # count the instances of  spam
    num_spam=df[df['labels']=='spam'].shape[0]
    print(num_spam)

    #randomly saple the ham instance to match  the nuber of of  spam instance
    ham_subset=df[df['labels']=='ham'].sample(num_spam,random_state=123)

    # cmine ham with spam 
    blanced_df=pd.concat([ham_subset,df[df['labels']=='spam']])

    return blanced_df

blanced_df=crate_blanced_dataset(df)
print(blanced_df.value_counts('labels'))
print(blanced_df.head())

747
labels
ham     747
spam    747
Name: count, dtype: int64
     labels                                               text
4307    ham  Awww dat is sweet! We can think of something t...
4138    ham                             Just got to  &lt;#&gt;
4831    ham  The word "Checkmate" in chess comes from the P...
4461    ham  This is wishing you a great day. Moji told me ...
5440    ham      Thank you. do you generally date the brothas?


In [13]:
blanced_df["labels"] = blanced_df["labels"].map({"ham": 0, "spam": 1})

In [14]:
crate a random _spit function the datataset into 3 parts 70%for training and  20 for  testing and 10 for vak=lidation

SyntaxError: invalid syntax (2110765754.py, line 1)

In [15]:
def random_split(df,train_frac,validation_frac):

    df =df.sample(frac=1,random_state=123).reset_index(drop=True)

    # calculatesplit indicees
    train_end=int(len(df)*train_frac)
    validation_end=train_end+int(len(df)*validation_frac)

    # split thed datatframe 
    train_df=df[:train_end]
    validation_df=df[train_end:validation_end]
    test_df=df[validation_df:]

    return train_df,validation_df,test_df
    

In [16]:
def random_split(df, train_frac, validation_frac):
    # Shuffle the entire DataFrame
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    # Calculate split indices
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    # Split the DataFrame
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(blanced_df, 0.7, 0.1)
# Test size is implied to be 0.2 as the remainder


In [17]:
print(len(train_df))
print(len(validation_df))
print(len(test_df))

1045
149
300


In [11]:
# now save the datat as csv
train_df.to_csv('train.csv',index=None)
validation_df.to_csv('validation.csv',index=None)
test_df.to_csv('test.csv',index=None)

In [18]:
import torch
from torch.utils.data import DataLoader,Dataset

class sapledataset(Dataset):
    def __init__(self,csv_file,tokenizer,max_length=None,pad_token_id=50256):
        self.data=pd.read_csv(csv_file)
        # pretokenized text

        self.encoded_texts=[tokenizer.encode(text) for text in self.data['text']]

        if max_length is None:
            self.max_length=self._longest_encoded_length()

        else:
            self.max_length=max_length
            # truncat seqewence if there are longer than max_length
            self.encoded_texts= [encoded_text[:self.max_length]  for encoded_text in self.encoded_texts]

        # pad seqence to the longest seqence
        self.encoded_texts =[encoded_text+[pad_token_id]*(self.max_length-len(encoded_text)) for  encoded_text in self.encoded_texts]

    def __getitem__(self,index):
        encoded=self.encoded_texts[index]  
        label=self.data.iloc[index]['labels']
        return(torch.tensor(encoded,dtype=torch.long),
              torch.tensor(label,dtype=torch.long))

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length=0
        for encoded_text  in self.encoded_texts:
            encode_length=len(encoded_text)
            if encode_length> max_length:
                max_length=encode_length

        return max_length
        
        

In [19]:
import tiktoken

tokenizer=tiktoken.get_encoding('gpt2')

In [20]:
train_dataset=sapledataset(csv_file='train.csv',
                          max_length=None,
                          tokenizer=tokenizer,
                          pad_token_id=50256)
print(train_dataset.max_length)

120


In [21]:
print(train_dataset)

In [22]:
train_dataset=sapledataset(csv_file='train.csv',
                          max_length=None,
                          tokenizer=tokenizer,
                          pad_token_id=50256)
print(len(train_dataset))

1045


In [23]:
val_dataset=sapledataset(csv_file='train.csv',max_length=train_dataset.max_length,
                        tokenizer=tokenizer)
test_dataset=sapledataset(csv_file='test.csv',max_length=train_dataset.max_length,tokenizer=tokenizer)

In [24]:
print('train loader\n:')
for input_batch,targrt_batch in train_loader:
    pass

print('input batch dimention:',input_batch.shape)

print('label batch dimention:',targrt_batch.shape)

train loader
:


NameError: name 'train_loader' is not defined

In [25]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

NameError: name 'train_loader' is not defined

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# This downloads the weights (approx. 500MB for the base version)
model_name = "openai-community/gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

print("GPT-2 weights downloaded and loaded successfully!")


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

C:\Users\PC\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\.cache\huggingface\hub\models--openai-community--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

In [21]:
import transformers

In [22]:
from transformers import GPT2LMHeadModel

In [25]:
print(GPT2LMHeadModel)

<class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>
